In [0]:
%pip install mlflow[databricks]==2.14.3 langchain==0.1.5 databricks-vectorsearch==0.22 databricks-sdk==0.18.0 flask_cors
dbutils.library.restartPython()

In [0]:
import os

host = "https://" + spark.conf.get("spark.databricks.workspaceUrl")
os.environ["databricks_token"] = dbutils.secrets.get(scope="credenciais", key="databricks_token")

index_name="workspace.rag.docs_idx"
host = "https://" + spark.conf.get("spark.databricks.workspaceUrl")

VECTOR_SEARCH_ENDPOINT_NAME="doc_vector_endpoint"

In [0]:
from databricks.vector_search.client import VectorSearchClient
from langchain_community.vectorstores import DatabricksVectorSearch
from langchain_community.embeddings import DatabricksEmbeddings
from databricks.sdk import WorkspaceClient

embedding_model = DatabricksEmbeddings(endpoint="databricks-qwen3-embedding-0-6b")

def get_retriever(persist_dir: str = None):
    # Use Databricks SDK built-in auth (works in both notebook and serving container)
    w = WorkspaceClient()
    host = w.config.host
    token = w.config.authenticate()["Authorization"].split(" ", 1)[1]

    vsc = VectorSearchClient(workspace_url=host, personal_access_token=token)
    vs_index = vsc.get_index(
        endpoint_name="doc_vector_endpoint",
        index_name="workspace.rag.docs_idx"
    )

    vectorstore = DatabricksVectorSearch(
        vs_index, text_column="text", embedding=embedding_model
    )
    return vectorstore.as_retriever()


In [0]:
import os
from databricks.sdk import WorkspaceClient

# Override secret token with notebook built-in auth (fixes 401 on Vector Search API)
_w = WorkspaceClient()
os.environ["databricks_token"] = _w.config.authenticate()["Authorization"].split(" ", 1)[1]

from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_community.chat_models import ChatDatabricks

chat_model = ChatDatabricks(endpoint="databricks-dbrx-instruct", max_tokens = 200)

TEMPLATE = """You are an AI business intelligence assistant for LH Nautical's executive team and stakeholders. Your task is to answer questions regarding predictive demand forecasting, sales performance, model evaluations (Baseline Moving Averages vs. XGBoost), distribution center (CD) demand growth, logistics expansion strategies, inventory health, and management report results using only the provided context. If the question is not related to LH Nautical's business, predictive analytics, distribution centers, or financial management reports, kindly decline to answer. If you don't know the answer or if the context does not contain enough information to respond accurately, simply state that you do not have that information in the available documentation—do not attempt to fabricate or extrapolate an answer. Keep the answers as concise, objective, and executive-ready as possible. Provide all answers strictly in Portuguese (PT-BR). Use the following pieces of context to answer the question at the end: {context} Question: {question} Answer:
"""
prompt = PromptTemplate(template=TEMPLATE, input_variables=["context", "question"])

chain = RetrievalQA.from_chain_type(
    llm=chat_model,
    chain_type="stuff",
    retriever=get_retriever(),
    chain_type_kwargs={"prompt": prompt}
)


In [0]:
# Fix: databricks-dbrx-instruct endpoint no longer exists; use an available one
chat_model = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct", max_tokens=200)
chain = RetrievalQA.from_chain_type(
    llm=chat_model,
    chain_type="stuff",
    retriever=get_retriever(),
    chain_type_kwargs={"prompt": prompt}
)

question = {"query": "Precisamos construir mais centros de distribuição? Se sim, onde?"}
answer = chain.run(question)
print(answer)

In [0]:
from mlflow.models import infer_signature
import mlflow
import langchain

mlflow.set_registry_uri("databricks-uc")
model_name = "workspace.rag.appliance_chatbot_model"

with mlflow.start_run(run_name="LH_Nautical_Chatbot") as run:
    signature = infer_signature(question, answer)
    model_info = mlflow.langchain.log_model(
        chain,
        loader_fn=get_retriever,  
        artifact_path="chain",
        registered_model_name=model_name,
        pip_requirements=[
            "mlflow==" + mlflow.__version__,
            "langchain==" + langchain.__version__,
            "databricks-vectorsearch",
            "flask_cors",
        ],
        input_example=question,
        signature=signature
    )